In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.integrate import solve_ivp
from joblib import Parallel, delayed
import logging
import os
import h5py
from tqdm import tqdm
import time
import traceback

# Configure logging
logging.basicConfig(
    filename="simulation_logs.txt",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger()

# Constants
g = 9.81  # gravity (m/s^2)
m = 70    # mass of the pendulum (kg)
L = 0.85  # length of the pendulum (m)
I = m * L**2 / 3  # moment of inertia

# PD controller gains
Kp = 10.0  # Proportional gain
Kd = 20.0  # Derivative gain

# Simulation parameters
t_eval = np.linspace(0, 25, 300)
time_step = t_eval[1] - t_eval[0]

# Parameter spaces
angles = [-1.5, -1.2, -0.9, -0.6, -0.3, 0, 0.3, 0.6, 0.9, 1.2, 1.5]
velocities = [-3, -2.4, -1.8, -1.2, -0.6, 0, 0.6, 1.2, 1.2, 2.4, 3]
initial_conditions = [(angle, velocity) for angle in angles for velocity in velocities]

noise_levels = np.round(np.arange(0, 3.1, 0.3), decimals=1)
delays = np.round(np.arange(0, 2.1, 0.2), decimals=1)
torque_levels = np.round(np.arange(25, 326, 50), decimals=0)

# Enhanced checkpoint system
class CheckpointManager:
    def __init__(self, filename):
        self.filename = filename
        self.temp_filename = filename + '.temp'
        self.progress_file = filename + '.progress'
        self.lock_file = filename + '.lock'
        
    def save_checkpoint(self, results, current_params):
        """Save results and current parameters atomically with retries"""
        max_attempts = 5
        for attempt in range(max_attempts):
            try:
                # Create lock file
                with open(self.lock_file, 'w') as lf:
                    lf.write(str(os.getpid()))
                
                # Save to temporary file first
                with h5py.File(self.temp_filename, 'w') as f:
                    # Save metadata
                    f.attrs['version'] = '1.0'
                    f.attrs['timestamp'] = time.time()
                    
                    # Save current simulation state
                    f.attrs['current_delay'] = current_params['delay']
                    f.attrs['current_torque'] = current_params['torque']
                    f.attrs['current_noise'] = current_params['noise']
                    
                    # Save results with compression
                    for idx, res in enumerate(results):
                        grp = f.create_group(f'sim_{idx}')
                        for key, val in res.items():
                            if isinstance(val, np.ndarray):
                                grp.create_dataset(
                                    key,
                                    data=val,
                                    compression='gzip',
                                    compression_opts=4
                                )
                            else:
                                grp.attrs[key] = val
                
                # Atomic rename
                if os.path.exists(self.filename):
                    os.remove(self.filename)
                os.rename(self.temp_filename, self.filename)
                
                # Save progress separately for redundancy
                with open(self.progress_file, 'w') as pf:
                    pf.write(f"{current_params['delay']},{current_params['torque']},{current_params['noise']}")
                
                # Remove lock file
                if os.path.exists(self.lock_file):
                    os.remove(self.lock_file)
                
                return True
                
            except Exception as e:
                logger.error(f"Checkpoint save attempt {attempt + 1} failed: {str(e)}")
                time.sleep(2 ** attempt)  # Exponential backoff
                
                # Clean up if failed
                if os.path.exists(self.temp_filename):
                    os.remove(self.temp_filename)
                if os.path.exists(self.lock_file):
                    os.remove(self.lock_file)
        
        logger.error(f"Failed to save checkpoint after {max_attempts} attempts")
        return False
    
    def load_checkpoint(self):
        """Load results with comprehensive error handling"""
        try:
            # Check for stale lock file
            if os.path.exists(self.lock_file):
                with open(self.lock_file, 'r') as lf:
                    pid = lf.read().strip()
                logger.warning(f"Found stale lock file from process {pid} - attempting recovery")
                os.remove(self.lock_file)
            
            # Try lightweight progress file first
            progress_data = None
            if os.path.exists(self.progress_file):
                try:
                    with open(self.progress_file, 'r') as pf:
                        parts = pf.read().strip().split(',')
                        if len(parts) == 3:
                            progress_data = {
                                'delay': float(parts[0]),
                                'torque': float(parts[1]),
                                'noise': float(parts[2])
                            }
                except Exception as e:
                    logger.warning(f"Error reading progress file: {str(e)}")
            
            # Try loading HDF5 file
            hdf5_data = None
            if os.path.exists(self.filename):
                try:
                    with h5py.File(self.filename, 'r') as f:
                        # Verify file structure
                        if 'sim_0' not in f:
                            raise ValueError("No simulation data found in file")
                        
                        # Get parameters
                        current_params = {
                            'delay': f.attrs.get('current_delay', delays[0]),
                            'torque': f.attrs.get('current_torque', torque_levels[0]),
                            'noise': f.attrs.get('current_noise', noise_levels[0])
                        }
                        
                        # Load results
                        results = []
                        for sim_name in sorted(f.keys()):
                            if sim_name.startswith('sim_'):
                                sim = f[sim_name]
                                res = {k: sim.attrs[k] for k in sim.attrs}
                                res.update({k: sim[k][()] for k in sim.keys()})
                                results.append(res)
                        
                        hdf5_data = (current_params, results)
                except Exception as e:
                    logger.error(f"Error loading HDF5 file: {str(e)}")
                    # Move corrupted file aside
                    corrupted_name = f"{self.filename}.corrupted_{int(time.time())}"
                    os.rename(self.filename, corrupted_name)
                    logger.info(f"Moved corrupted file to {corrupted_name}")
            
            # Determine what to return
            if hdf5_data:
                return hdf5_data
            elif progress_data:
                return progress_data, []
            else:
                return None, []
                
        except Exception as e:
            logger.error(f"Critical checkpoint error: {str(e)}\n{traceback.format_exc()}")
            return None, []

# Simulation functions
def add_noise(value, stddev):
    return value + np.random.normal(0, stddev)

def PD_controller(theta, omega, Kp, Kd, A, delay, previous_states, time_step, noise_stddev_theta, noise_stddev_omega):
    if len(previous_states) > 0:
        delay_index = max(0, len(previous_states) - int(delay / time_step))
        observed_theta, observed_omega = previous_states[delay_index]
    else:
        observed_theta, observed_omega = theta, omega  # No delay initially
    
    observed_theta = add_noise(observed_theta, noise_stddev_theta)
    observed_omega = add_noise(observed_omega, noise_stddev_omega)
    torque = -Kp * observed_theta - Kd * observed_omega
    
    # Discretizing torque levels
    if torque > 0:
        torque = A
    elif torque < 0:
        torque = -A
    else:
        torque = 0
    return torque

def pendulum(t, state, previous_states, time_step, Kp, Kd, A, delay, noise_stddev_theta, noise_stddev_omega):
    theta, omega = state
    torque = PD_controller(theta, omega, Kp, Kd, A, delay, previous_states, time_step, noise_stddev_theta, noise_stddev_omega)
    
    previous_states.append([theta, omega])
    if len(previous_states) > int(delay / time_step):
        previous_states.pop(0)
    
    return [omega, (-m * g * L * np.sin(theta) + torque) / I]

def calculate_settling_time(signal, t, tolerance=0.05):
    steady_state_value = 0
    tolerance_band = tolerance * abs(signal[0])
    lower_bound = steady_state_value - tolerance_band
    upper_bound = steady_state_value + tolerance_band
    
    within_band = (signal >= lower_bound) & (signal <= upper_bound)
    
    for i in range(len(signal)):
        if np.all(within_band[i:]):
            return t[i]
    return np.nan

def run_simulation(delay, A, noise_stddev, initial_state):
    previous_states = []
    try:
        sol = solve_ivp(
            pendulum,
            [t_eval[0], t_eval[-1]],
            initial_state,
            t_eval=t_eval,
            args=(previous_states, time_step, Kp, Kd, A, delay, noise_stddev, noise_stddev),
            method='RK45',      # Auto-switches between stiff/non-stiff
            rtol=1e-3,          # Relative tolerance
            atol=1e-4,          # Absolute tolerance
            first_step=1e-3,    # Initial step size (helps with torque discontinuities)
            max_step=0.1,       # Prevents overly large steps during stable phases
            vectorized=True     # Enable if your PD_controller supports vector inputs
        )
        
        # Calculate torque history
        torque = np.array([
            PD_controller(t, o, Kp, Kd, A, delay, previous_states, time_step, noise_stddev, noise_stddev)
            for t, o in zip(sol.y[0], sol.y[1])
        ])
        
        settling_time = calculate_settling_time(sol.y[0], sol.t)
        
        return {
            'time': sol.t,
            'theta': sol.y[0],
            'omega': sol.y[1],
            'torque': torque,
            'settling_time': settling_time
        }
    except Exception as e:
        logger.error(f"Simulation failed for delay={delay}, A={A}, noise={noise_stddev}: {str(e)}")
        return {
            'time': np.array([]),
            'theta': np.array([]),
            'omega': np.array([]),
            'torque': np.array([]),
            'settling_time': np.nan
        }

# Main simulation loop
def run_all_simulations(checkpoint_file="checkpoint.h5"):
    # Clean up any corrupted state
    for fname in [checkpoint_file, checkpoint_file + '.temp', checkpoint_file + '.progress']:
        if os.path.exists(fname + '.corrupted'):
            os.remove(fname + '.corrupted')
    
    checkpoint = CheckpointManager(checkpoint_file)
    
    # Try to load checkpoint
    current_params, full_results = checkpoint.load_checkpoint()
    
    if current_params is None:
        # Start from beginning if no checkpoint found
        current_params = {
            'delay': delays[0],
            'torque': torque_levels[0],
            'noise': noise_levels[0]
        }
        logger.info("Starting new simulation from beginning")
    else:
        logger.info(f"Resuming simulation from: delay={current_params['delay']}, "
                   f"torque={current_params['torque']}, noise={current_params['noise']}")
    
    # Find starting indices
    try:
        delay_idx = np.where(delays == current_params['delay'])[0][0]
        A_idx = np.where(torque_levels == current_params['torque'])[0][0]
        noise_idx = np.where(noise_levels == current_params['noise'])[0][0]
    except IndexError:
        logger.error("Parameter mismatch - starting from beginning")
        delay_idx, A_idx, noise_idx = 0, 0, 0
    
    # Main simulation loop
    try:
        for delay_idx in range(delay_idx, len(delays)):
            delay = delays[delay_idx]
            for A_idx in range(A_idx, len(torque_levels)):
                A = torque_levels[A_idx]
                for noise_idx in range(noise_idx, len(noise_levels)):
                    noise_stddev = noise_levels[noise_idx]
                    
                    # Update current parameters
                    current_params.update({
                        'delay': delay,
                        'torque': A,
                        'noise': noise_stddev
                    })
                    
                    logger.info(f"Running: Delay={delay:.1f}s, Torque={A}Nm, Noise={noise_stddev:.1f}")
                    
                    # Run simulations with progress bar
                    try:
                        with tqdm(total=len(initial_conditions), desc=f"Delay={delay}, Torque={A}") as pbar:
                            sim_results = Parallel(n_jobs=-1)(
                                delayed(run_simulation)(delay, A, noise_stddev, ic)
                                for ic in initial_conditions
                            )
                            pbar.update(len(initial_conditions))
                    except Exception as e:
                        logger.error(f"Error in parallel execution: {str(e)}\n{traceback.format_exc()}")
                        continue
                    
                    # Store results
                    for ic_idx, res in enumerate(sim_results):
                        result = {
                            'delay': delay,
                            'torque': A,
                            'noise': noise_stddev,
                            'ic_idx': ic_idx,
                            'ic_theta': initial_conditions[ic_idx][0],
                            'ic_omega': initial_conditions[ic_idx][1],
                            'time': res['time'],
                            'theta': res['theta'],
                            'omega': res['omega'],
                            'torque_history': res['torque'],
                            'settling_time': res['settling_time']
                        }
                        full_results.append(result)
                    
                    # Save checkpoint after each parameter combination
                    if not checkpoint.save_checkpoint(full_results, current_params):
                        logger.error("Failed to save checkpoint - stopping simulation")
                        return full_results
                    
                    # Reset noise index after first complete run
                    noise_idx = 0
                
                # Reset torque index after first complete run
                A_idx = 0
            
            # Reset delay index after first complete run
            delay_idx = 0
        
        return full_results
        
    except KeyboardInterrupt:
        logger.info("Simulation interrupted by user - saving final checkpoint")
        checkpoint.save_checkpoint(full_results, current_params)
        return full_results
    except Exception as e:
        logger.error(f"Unexpected error: {str(e)}\n{traceback.format_exc()}")
        checkpoint.save_checkpoint(full_results, current_params)
        raise

def generate_heatmaps(hdf5_file):
    """Generate heatmaps from simulation results"""
    os.makedirs("heatmaps", exist_ok=True)
    
    try:
        with h5py.File(hdf5_file, 'r') as f:
            # Load all data into DataFrame
            data = []
            for sim in f.values():
                if not isinstance(sim, h5py.Group):
                    continue
                
                data.append({
                    'delay': sim.attrs['delay'],
                    'torque': sim.attrs['torque'],
                    'noise': sim.attrs['noise'],
                    'ic_theta': sim.attrs['ic_theta'],
                    'ic_omega': sim.attrs['ic_omega'],
                    'settling_time': sim.attrs['settling_time']
                })
        
        df = pd.DataFrame(data)
        
        # Create averaged heatmaps
        plt.figure(figsize=(24, 8))
        
        # 1. Noise vs Delay (averaged)
        plt.subplot(1, 3, 1)
        pivot1 = df.pivot_table(values='settling_time', index='noise', columns='delay', aggfunc=np.mean)
        sns.heatmap(pivot1, annot=True, fmt=".2f", cmap="viridis", 
                    cbar_kws={'label': 'Average Settling Time (s)'})
        plt.title("Average Noise vs Delay\nAcross All Initial Conditions")
        
        # 2. Noise vs Torque (averaged)
        plt.subplot(1, 3, 2)
        pivot2 = df.pivot_table(values='settling_time', index='noise', columns='torque', aggfunc=np.mean)
        sns.heatmap(pivot2, annot=True, fmt=".2f", cmap="viridis",
                   cbar_kws={'label': 'Average Settling Time (s)'})
        plt.title("Average Noise vs Torque\nAcross All Initial Conditions")
        
        # 3. Delay vs Torque (averaged)
        plt.subplot(1, 3, 3)
        pivot3 = df.pivot_table(values='settling_time', index='delay', columns='torque', aggfunc=np.mean)
        sns.heatmap(pivot3, annot=True, fmt=".2f", cmap="viridis",
                   cbar_kws={'label': 'Average Settling Time (s)'})
        plt.title("Average Delay vs Torque\nAcross All Initial Conditions")
        
        plt.tight_layout()
        plt.savefig(f"heatmaps/averaged_heatmaps.png", dpi=300, bbox_inches='tight')
        plt.close()
        
        # Create heatmaps for each IC
        for ic_idx, ic_data in df.groupby(['ic_theta', 'ic_omega']):
            theta0, omega0 = ic_idx
            plt.figure(figsize=(24, 8))
            
            # 1. Noise vs Delay
            plt.subplot(1, 3, 1)
            pivot1 = ic_data.pivot_table(values='settling_time', index='noise', columns='delay')
            sns.heatmap(pivot1, annot=True, fmt=".2f", cmap="viridis", 
                        cbar_kws={'label': 'Settling Time (s)'})
            plt.title(f"Noise vs Delay\nθ₀={theta0:.1f}, ω₀={omega0:.1f}")
            
            # 2. Noise vs Torque
            plt.subplot(1, 3, 2)
            pivot2 = ic_data.pivot_table(values='settling_time', index='noise', columns='torque')
            sns.heatmap(pivot2, annot=True, fmt=".2f", cmap="viridis",
                       cbar_kws={'label': 'Settling Time (s)'})
            plt.title(f"Noise vs Torque\nθ₀={theta0:.1f}, ω₀={omega0:.1f}")
            
            # 3. Delay vs Torque
            plt.subplot(1, 3, 3)
            pivot3 = ic_data.pivot_table(values='settling_time', index='delay', columns='torque')
            sns.heatmap(pivot3, annot=True, fmt=".2f", cmap="viridis",
                       cbar_kws={'label': 'Settling Time (s)'})
            plt.title(f"Delay vs Torque\nθ₀={theta0:.1f}, ω₀={omega0:.1f}")
            
            plt.tight_layout()
            plt.savefig(f"heatmaps/ic_{theta0:.1f}_{omega0:.1f}_heatmaps.png", dpi=300, bbox_inches='tight')
            plt.close()
            
    except Exception as e:
        logger.error(f"Error generating heatmaps: {str(e)}\n{traceback.format_exc()}")
        raise

if __name__ == "__main__":
    try:
        # Clean up any corrupted files from previous runs
        for fname in ['checkpoint.h5', 'final_results.h5']:
            if os.path.exists(fname + '.corrupted'):
                os.remove(fname + '.corrupted')
        
        # Run simulations
        results = run_all_simulations()
        
        # Save final results
        final_checkpoint = CheckpointManager("final_results.h5")
        if final_checkpoint.save_checkpoint(results, {
            'delay': delays[-1],
            'torque': torque_levels[-1],
            'noise': noise_levels[-1]
        }):
            print("\nSimulation completed successfully!")
            print(f"Results saved to: final_results.h5")
            
            # Generate heatmaps
            generate_heatmaps("final_results.h5")
            print(f"Heatmaps saved to: heatmaps/ directory")
        
    except Exception as e:
        print(f"\nSimulation failed with error: {str(e)}")
        print("Check simulation_logs.txt for details")
        logger.error(f"Main execution failed: {str(e)}\n{traceback.format_exc()}")